In [ ]:
%%bash
apt-get -y -qq update
apt-get -y -qq install postgresql postgresql-contrib >/dev/null
service postgresql start
sudo -u postgres psql -qc "DROP DATABASE IF EXISTS taxi;"
sudo -u postgres psql -qc "DROP ROLE IF EXISTS taxi;"
sudo -u postgres psql -qc "CREATE ROLE taxi LOGIN PASSWORD 'taxi' SUPERUSER;"
sudo -u postgres psql -qc "CREATE DATABASE taxi OWNER taxi;"
sudo -u postgres psql -tAc "SHOW server_version;"
echo "postgres up"


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ── config ──────────────────────────────────────────────────────────────────
RAW_TRAIN   = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/cleaned_2015_2016.parquet"
OUT_SERIES  = "/content/drive/MyDrive/Timesfm_Chronos_fine-tune/data/taxi_series.parquet"
ENG_TRAIN   = "/content/engineered_2015_2016.parquet"   # local disk
N_TEST_DAYS = 120        # last N days → holdout (val). 2017 file is ignored.

PG = dict(host="127.0.0.1", port=5432, dbname="taxi", user="taxi", password="taxi")
PG_DSN = "dbname={dbname} user={user} password={password} host={host} port={port}".format(**PG)
DT_COL, VENDOR_COL = "tpep_pickup_datetime", "vendorid"
assert os.path.exists(RAW_TRAIN), "fix RAW_TRAIN"


In [ ]:
!pip -q install polars pyarrow duckdb psycopg2-binary
import polars as pl, pyarrow.parquet as pq, time

def engineer(src, dst):
    names = pq.ParquetFile(src).schema_arrow.names
    lf = pl.scan_parquet(src).rename({n: n.lower() for n in names})
    lf = lf.with_columns([
        ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime"))
            .dt.total_seconds() / 60).alias("trip_duration_minutes"),
        pl.col("tpep_pickup_datetime").dt.hour().alias("pickup_hour"),
        pl.col("tpep_pickup_datetime").dt.date().alias("pickup_date"),
    ]).with_columns([
        (pl.col("trip_distance") / pl.when(pl.col("trip_duration_minutes") > 0)
            .then(pl.col("trip_duration_minutes") / 60).otherwise(0.001)).alias("trip_speed_mph"),
    ]).with_columns([
        (pl.col("total_amount") / pl.when(pl.col("trip_distance") > 0)
            .then(pl.col("trip_distance")).otherwise(0.001)).alias("price_per_distance"),
    ])
    g = (lf.group_by(["pickup_date", "pickup_hour"])
           .agg([pl.len().alias("hourly_trip_volume"),
                 pl.col("trip_speed_mph").mean().alias("hourly_avg_speed")]))
    try:    agg = g.collect(streaming=True)
    except TypeError: agg = g.collect(engine="streaming")
    (lf.join(agg.lazy(), on=["pickup_date", "pickup_hour"], how="left")
       .drop(["pickup_date", "pickup_hour"]).sink_parquet(dst))

t0 = time.time(); engineer(RAW_TRAIN, ENG_TRAIN)
print(f"engineered in {time.time()-t0:.0f}s")
print(pl.read_parquet(ENG_TRAIN, n_rows=3).select(
    ["vendorid","tpep_pickup_datetime","trip_distance","trip_duration_minutes",
     "trip_speed_mph","price_per_distance","hourly_trip_volume","hourly_avg_speed"]))


In [ ]:
import duckdb, psycopg2, pandas as pd, time
con = duckdb.connect()
con.execute("INSTALL postgres; LOAD postgres;")
con.execute(f"ATTACH '{PG_DSN}' AS pg (TYPE postgres);")
con.execute("DROP TABLE IF EXISTS pg.sefer_egitim;")
t0 = time.time()
con.execute(f"CREATE TABLE pg.sefer_egitim AS SELECT * FROM read_parquet('{ENG_TRAIN}');")
n = con.execute("SELECT count(*) FROM pg.sefer_egitim").fetchone()[0]
con.close(); print(f"sefer_egitim: {n:,} rows in {time.time()-t0:.0f}s")

def cols(table):
    with psycopg2.connect(PG_DSN) as c:
        return pd.read_sql("SELECT column_name FROM information_schema.columns "
                           "WHERE table_name=%s ORDER BY ordinal_position", c, params=[table])
print(sorted(cols("sefer_egitim").column_name.tolist()))


In [ ]:
import psycopg2
with psycopg2.connect(PG_DSN) as c, c.cursor() as cur:
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_dt  ON sefer_egitim ("{DT_COL}");')
    cur.execute(f'CREATE INDEX IF NOT EXISTS idx_dtv ON sefer_egitim ("{DT_COL}","{VENDOR_COL}");')
    cur.execute("ANALYZE sefer_egitim;")
    c.commit()
print("indexed.")

In [ ]:
METRICS = ["sefer","passenger_count","trip_distance","fare_amount","tip_amount",
           "tolls_amount","total_amount","trip_duration_minutes","trip_speed_mph",
           "price_per_distance","hourly_trip_volume","hourly_avg_speed"]
RES_SECONDS = {"1m":60,"3m":180,"5m":300,"10m":600,"15m":900,"30m":1800,
               "1h":3600,"3h":10800,"1d":86400}
VENDORS = ["hepsi","1","2"]
present = set(cols("sefer_egitim").column_name)
AVG_METRICS = [m for m in METRICS[1:] if m in present]
print("metrics used:", ["sefer"]+AVG_METRICS, "| missing:", [m for m in METRICS[1:] if m not in present] or "none")


In [ ]:
import psycopg2, pandas as pd
parts = []
with psycopg2.connect(PG_DSN) as c:
    for vendor in VENDORS:
        for res, dk in RES_SECONDS.items():
            where  = "" if vendor=="hepsi" else f'WHERE "{VENDOR_COL}"=%s'
            params = [] if vendor=="hepsi" else [vendor]
            med = ",\n".join(f'percentile_cont(0.5) WITHIN GROUP (ORDER BY "{m}") AS "{m}"'
                             for m in AVG_METRICS)
            sql = f'''SELECT date_bin(INTERVAL '{dk} seconds', "{DT_COL}",
                             TIMESTAMP '1970-01-01 00:00:00') AS ts,
                             count(*) AS sefer{"," if med else ""}
                             {med}
                      FROM sefer_egitim {where} GROUP BY ts ORDER BY ts'''
            d = pd.read_sql(sql, c, params=params)
            if len(d)==0: continue
            d["ts"] = pd.to_datetime(d["ts"])
            for metric in ["sefer"]+AVG_METRICS:
                sub = d[["ts",metric]].dropna(subset=[metric]).rename(columns={metric:"value"})
                if len(sub)==0: continue
                sub["series_id"]=f"{metric}__{res}__{vendor}"; sub["metric"]=metric
                sub["resolution"]=res; sub["vendor"]=vendor
                parts.append(sub[["series_id","metric","resolution","vendor","ts","value"]])
            print(f"  {vendor} {res}: {len(d):,} bins", flush=True)
series = pd.concat(parts, ignore_index=True)
print("total binned rows:", f"{len(series):,}")